# Imports

In [1]:
from datasets import load_dataset
import spacy 
from spacy.tokens import DocBin

## Daten Laden und Anschauen

In [ ]:
dataset = load_dataset("jfrei/GPTNERMED", trust_remote_code=True)

In [3]:
print(dataset)
print(dataset["train"].features)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['sentence', 'ner_labels'],
        num_rows: 7876
    })
    validation: Dataset({
        features: ['sentence', 'ner_labels'],
        num_rows: 984
    })
    test: Dataset({
        features: ['sentence', 'ner_labels'],
        num_rows: 985
    })
})
{'sentence': Value(dtype='string', id=None), 'ner_labels': Sequence(feature={'ner_class': ClassLabel(names=['Medikation', 'Dosis', 'Diagnose'], id=None), 'start': Value(dtype='int32', id=None), 'stop': Value(dtype='int32', id=None)}, length=-1, id=None)}
{'sentence': '0,4 Diuretika 0,25 1x/die', 'ner_labels': {'ner_class': [1, 0, 1, 1], 'start': [0, 4, 14, 19], 'stop': [3, 13, 18, 25]}}


## Scapy 

In [4]:
example = dataset["train"][0]
text = example["sentence"]
label = example["ner_labels"]
print(text)
print(label)

0,4 Diuretika 0,25 1x/die
{'ner_class': [1, 0, 1, 1], 'start': [0, 4, 14, 19], 'stop': [3, 13, 18, 25]}


In [23]:
import spacy
from spacy.tokens import DocBin
from spacy.util import filter_spans

def get_label(label_id):
    label_mapping = {
        0: "Medikation",
        1: "Dosis",
        2: "Diagnose",
    }
    return label_mapping.get(label_id)

def data_to_docbin(data):
    doc_bin = DocBin()
    nlp = spacy.blank("de")

    for example in data:
        text = example["sentence"]
        labels = example["ner_labels"]
        doc = nlp(text)

        ents = []
        for start, stop, label_id in zip (labels["start"], labels["stop"], labels["ner_class"]):

            label = get_label(label_id)
            span = doc.char_span(int(start), int(stop), label=label, alignment_mode="expand")
            ents.append(span)
            filtered_ents = filter_spans(ents)
        
        doc.ents = filtered_ents
        doc_bin.add(doc)

    return doc_bin

In [ ]:
doc_bin = data_to_docbin(dataset["train"])
doc_bin.to_disk("./train.spacy")

doc_bin = data_to_docbin(dataset["validation"])
doc_bin.to_disk("./validation.spacy")

doc_bin = data_to_docbin(dataset["test"])
doc_bin.to_disk("./test.spacy")

# Model Training
- first for deep/gbert-large

### Erstelle Config

- hier allgemein, beim training weiter anpassen

In [1]:
%%bash
python -m spacy init config config_spacy.cfg \
  --lang de --pipeline ner --optimize accuracy --gpu --force
    

ℹ Generated config template specific for your use case
- Language: de
- Pipeline: ner
- Optimize for: accuracy
- Hardware: GPU
- Transformer: bert-base-german-cased
✔ Auto-filled config with all values
✔ Saved config
config_spacy.cfg
You can now add your data and train your pipeline:
python -m spacy train config_spacy.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


## Trainiere Modell

In [5]:
import sys

!{sys.executable} -m spacy train config.cfg \
    --output models/gbert-large \
    --paths.train data_split/train.spacy \
    --paths.dev data_split/validation.spacy \
    --components.transformer.model.name deepset/gbert-large \
    --gpu-id -1

ℹ Saving to output directory: models/gbert-large
ℹ Using CPU
✘ Error parsing config overrides
components -> transformer -> model -> name	not a section value that can be overridden
